[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DucThinhmta/MentalHealthAI/blob/main/notebooks/run_app_colab.ipynb)

# MentalHealthAI: chạy cả ứng dụng trên Colab (không cần cài gì trên máy)

Mở notebook này, bấm **Runtime > Run all**, rồi mở liên kết in ra ở ô cuối. Mô hình, thư viện và dữ liệu đều nằm trên máy chủ Google;
máy của bạn chỉ cần trình duyệt. Dùng **GPU (T4)** nếu có (nhanh hơn khi phân tích), CPU vẫn chạy được.

> **Quyền riêng tư:** liên kết là địa chỉ công khai tạm thời. Ai có liên kết đều mở được. Chỉ gửi cho người trong nhóm, tắt notebook khi xong,
> và **không dùng để thu dữ liệu nghiên cứu từ người thật** khi chưa có phê duyệt đạo đức (xem `docs/nghien_cuu/`). Phiên Colab tự xoá khi đóng, nên trò chuyện không được lưu lại.

## 0. Chuẩn bị (ô nào cũng chạy trên máy chủ Google, không tải gì về máy bạn)

**Colab Secrets** (biểu tượng chìa khoá ở thanh bên trái, bật "Notebook access" cho từng khoá): dùng để cất khoá thay vì dán vào ô mã.
- `HF_TOKEN`: chỉ cần nếu 3 mô hình trên Hugging Face ở chế độ private.
- `GITHUB_TOKEN`: chỉ cần nếu repo GitHub ở chế độ private (quyền đọc).
- `ANTHROPIC_API_KEY`: tuỳ chọn, bật chế độ LLM (trò chuyện tự nhiên và dịch). Không có thì bot trả lời theo luật.

**Không bao giờ dán khoá vào ô mã hoặc chia sẻ notebook đã chứa khoá.**

In [ ]:
import os, subprocess, sys
REPO = "DucThinhmta/MentalHealthAI"
BRANCH = "main"            # đổi sang nhánh khác nếu muốn thử tính năng chưa gộp
sys.path.insert(0, "/content/MentalHealthAI/src")
if not os.path.exists("/content/MentalHealthAI"):
    tok = ""
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        pass
    url = f"https://{tok + '@' if tok else ''}github.com/{REPO}.git"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/MentalHealthAI"], check=True)
%cd /content/MentalHealthAI
import colab_helpers as ch
os.environ["HF_TOKEN"] = ch.secret("HF_TOKEN", "") or ""
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Thư viện nhẹ (Colab đã có sẵn torch, transformers, pandas...). Chỉ cài phần còn thiếu.
!pip -q install -r requirements-colab.txt

## 1. Mô hình
Tải 3 mô hình BERT từ Hugging Face vào **ổ đĩa tạm của Colab** (không về máy bạn). Lần đầu mất vài phút.
Để các lần sau nhanh hơn, có thể bật bước lưu vào Drive ở ô tuỳ chọn dưới (tốn dung lượng Drive).

In [ ]:
!python src/download_models.py

In [ ]:
# Tuỳ chọn: mô hình dịch Việt <-> Anh chạy trên máy chủ (dùng khi KHÔNG có ANTHROPIC_API_KEY để phân tích tiếng Việt chính xác hơn)
DOWNLOAD_TRANSLATOR = True
if DOWNLOAD_TRANSLATOR:
    !python src/translator.py --download opus --direction vi-en

## 2. Chạy ứng dụng và lấy liên kết

In [ ]:
import os, colab_helpers as ch
env = {}
key = ch.secret("ANTHROPIC_API_KEY")
if key:
    env["ANTHROPIC_API_KEY"] = key
    print("Chế độ LLM: BẬT (tin nhắn được gửi tới dịch vụ AI để tạo phản hồi và dịch)")
else:
    print("Chế độ LLM: TẮT (bot trả lời theo luật)")
app = ch.start_app(8501, env=env)
assert ch.wait_for_port(8501), "Ứng dụng chưa lên, xem lỗi bằng: !streamlit run src/app.py"
url, tunnel = ch.start_tunnel(8501)
print("\nMở ứng dụng tại:", url)

Giữ notebook mở trong lúc dùng (Colab ngắt khi bỏ không quá lâu). Khi xong: **Runtime > Disconnect and delete runtime**.
Nếu liên kết lỗi, chạy lại ô trên. Cập nhật mã: chạy lại từ ô "Chuẩn bị" với `BRANCH` mới (xoá thư mục cũ bằng `!rm -rf /content/MentalHealthAI` trước).